# litert-tunner — Quickstart: Fine-tune an INT8 Model

This notebook walks through the full workflow:

1. **Train** a small MobileNetV2 classifier on CIFAR-10
2. **Export** the trained model to a fully-quantized INT8 `.tflite` file
3. **Evaluate** the INT8 model and compare accuracy with the original float model
4. **Fine-tune** the INT8 model using `litert_tunner` to recover quantization accuracy loss
5. **Export** the fine-tuned INT8 model and verify accuracy improvement

> **Requirements:** Make sure the project is installed with dev dependencies:
> ```bash
> pip install -e ".[dev]"
> ```

In [ ]:
%load_ext autoreload
%autoreload 2

## 0. Imports & Setup

In [ ]:
import os
import tempfile
from pathlib import Path

import keras
import numpy as np
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

import litert_tunner

print(f"Keras version: {keras.__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"litert_tunner version: {litert_tunner.__version__}")

# Create a temp directory for model files
WORK_DIR = Path(tempfile.mkdtemp(prefix="litert_tunner_demo_"))
WORK_DIR = Path("/tmp/litert_tunner_demo__22c8y5t")
print(f"Working directory: {WORK_DIR}")

## 1. Load CIFAR-10 and Train a Float32 Model

We use CIFAR-10 (60k 32×32 color images, 10 classes) which downloads automatically
via `keras.datasets`. We build a small MobileNetV2-based classifier.

In [ ]:
# Load CIFAR-10
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train_full = x_train_full.astype(np.float32)
x_test = x_test.astype(np.float32)

y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

NUM_TRAIN = 10000
NUM_TEST = 1000

x_train = x_train_full[:NUM_TRAIN]
y_train = y_train_full[:NUM_TRAIN]
x_test = x_test[:NUM_TEST]
y_test = y_test[:NUM_TEST]

NUM_CLASSES = 10
INPUT_SHAPE = (32, 32, 3)

print(f"Train: {x_train.shape}, Test: {x_test.shape}")
print(f"Labels: {NUM_CLASSES} classes")

In [ ]:
keras.utils.set_random_seed(42)

backbone = keras.applications.MobileNetV3Small(
    include_top=False,
    # weights="imagenet",
    input_shape=INPUT_SHAPE,
    minimalistic=True,
)

inputs = backbone.input
x = backbone(inputs, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, name="logits")(x)

float_model = keras.Model(inputs=inputs, outputs=outputs, name="mobilenetv2_cifar10")
# float_model.summary()

In [ ]:
# Train the float model
float_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
    jit_compile=True
)

# history = float_model.fit(
#     x_train, y_train,
#     validation_data=(x_test, y_test),
#     epochs=2,
#     batch_size=16,
# )
# float_model.save(WORK_DIR / "model.keras")

In [ ]:
float_model.load_weights(WORK_DIR / "model.keras")

In [ ]:
float_loss, float_acc = float_model.evaluate(x_test, y_test)
print(f"\n✅ Float32 Test model accuracy: {float_acc:.4f}")

float_loss, float_acc = float_model.evaluate(x_train, y_train)
print(f"\n✅ Float32 Train model accuracy: {float_acc:.4f}")

## 2. Export to Fully-Quantized INT8 TFLite

We use `tf.lite.TFLiteConverter` with a representative dataset to produce
a fully-quantized INT8 model with float32 I/O.

In [ ]:
from tqdm import tqdm

def export_to_int8_tflite(model: keras.Model, output_path: Path, calibration_data: np.ndarray):
    """Export a Keras model to fully-quantized INT8 TFLite with float I/O."""
    def representative_dataset():
        rng = np.random.default_rng(42)
        indices = rng.choice(len(calibration_data), size=200, replace=False)
        for i in tqdm(indices, desc="Sampling ..."):
            yield [calibration_data[i : i + 1]]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    # Keep float32 I/O for easier evaluation
    tflite_bytes = converter.convert()

    output_path.write_bytes(tflite_bytes)
    size_kb = len(tflite_bytes) / 1024
    print(f"Exported INT8 model: {output_path.name} ({size_kb:.1f} KB)")


int8_model_path = WORK_DIR / "mobilenetv2_int8.tflite"
export_to_int8_tflite(float_model, int8_model_path, x_train)

## 3. Evaluate the INT8 Model

Run inference through the LiteRT Interpreter and compare accuracy with the
original float model. You'll typically see a small accuracy drop due to
quantization.

In [ ]:
def run_tflite_inference(model_path: Path, x_data: np.ndarray) -> np.ndarray:
    """Run batched inference through the LiteRT Interpreter."""
    interpreter = Interpreter(model_path=str(model_path), num_threads=4)
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    all_outputs = []
    for i in range(len(x_data)):
        sample = x_data[i:i+1].astype(np.float32)
        interpreter.resize_tensor_input(input_details[0]["index"], list(sample.shape))
        interpreter.allocate_tensors()
        interpreter.set_tensor(input_details[0]["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        all_outputs.append(output)

    return np.concatenate(all_outputs, axis=0)


# Run INT8 inference
int8_logits = run_tflite_inference(int8_model_path, x_test)
int8_preds = np.argmax(int8_logits, axis=-1)
int8_acc = np.mean(int8_preds == y_test)

print(f"Float32 model accuracy:  {float_acc:.4f}")
print(f"INT8 TFLite accuracy:    {int8_acc:.4f}")
print(f"Accuracy drop:           {float_acc - int8_acc:+.4f}")

## 4. Load with litert-tunner & Verify

Load the INT8 model using `litert_tunner.load_model()` which creates a Keras
replica with differentiable quantization simulation. The forward pass should
match the LiteRT Interpreter output.

In [ ]:
# Load the INT8 model as a trainable Keras model
tunner_model = litert_tunner.load_model(str(int8_model_path))

# Verify forward pass matches the Interpreter
tunner_logits = tunner_model.predict(x_test, verbose=0)
int8_logits_sample = run_tflite_inference(int8_model_path, x_test)
float32_logits_sample = float_model.predict(x_test, verbose=0)

# litert_tunner.assert_cosine_similarity(tunner_logits, int8_logits_sample, min_similarity=0.99)

# max_value = np.abs(int8_logits_sample).max()
# litert_tunner.assert_allclose_with_mismatch_tolerance(
#     tunner_logits / max_value,
#     int8_logits_sample / max_value,
#     atol=0.07,
#     max_mismatch_fraction=0.02,
# )

In [ ]:
from litert_tunner import testing_utils

tunner_vs_litert = testing_utils.cosine_similarity(tunner_logits, int8_logits_sample)
float32_vs_litert = testing_utils.cosine_similarity(float32_logits_sample, int8_logits_sample)
tunner_vs_litert, float32_vs_litert

In [ ]:
tunner_preds = np.argmax(tunner_logits, axis=-1)
tunner_acc = np.mean(tunner_preds == y_test)

print(f"Float32 model accuracy: {float_acc:.4f}")
print(f"INT8 TFLite accuracy  : {int8_acc:.4f}")
print(f"Tunner accuracy       : {tunner_acc:.4f}")

In [ ]:
tunner_model.layers[-2].weight_quant._scale_var

In [ ]:
tunner_model.layers[-2].trainable_variables

## 5. Fine-tune the INT8 Model

We use **distillation fine-tuning**: the original float model acts as the
teacher, and the quantized tunner model acts as the student. This helps
recover accuracy lost during quantization.

Only bias and scale parameters are unfrozen by default.

In [ ]:
# Prepare the tunner model for fine-tuning
# This freezes all weights except biases and weight scales
litert_tunner.prepare_for_finetuning(tunner_model, trainable_pattern=r".*(bias|weight_scale)$")

# Count trainable vs total parameters
total_params = sum(np.prod(v.shape) for v in tunner_model.variables)
trainable_params = sum(np.prod(v.shape) for v in tunner_model.trainable_variables)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

In [ ]:
# tunner_model.summary()

In [ ]:
# for layer in tunner_model.layers[-30:]:
#     for v in layer.variables:
#         # print(v.path)
#         # if "weight_scale" in v.path and "quantized_dense" in v.path:
#         if "weight_scale" in v.path:
#             print(v.path)
#             v.trainable = True


# Count trainable vs total parameters
total_params = sum(np.prod(v.shape) for v in tunner_model.variables)
trainable_params = sum(np.prod(v.shape) for v in tunner_model.trainable_variables)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

In [ ]:
import keras


def cosine_similarity(y_pred, y_true):
    # Flatten inputs to (batch_size, -1)
    y_pred_flat = keras.ops.reshape(y_pred, (keras.ops.shape(y_pred)[0], -1))
    y_true_flat = keras.ops.reshape(y_true, (keras.ops.shape(y_true)[0], -1))

    # Normalize vectors
    pred_norm = keras.ops.normalize(y_pred_flat, axis=-1)
    true_norm = keras.ops.normalize(y_true_flat, axis=-1)

    # Calculate average cosine similarity across the batch
    cosine_sim = keras.ops.mean(keras.ops.sum(pred_norm * true_norm, axis=-1))
    return cosine_sim


In [ ]:
trainer = litert_tunner.Trainer(
    litert_model=tunner_model,
    base_model=float_model,
    l2_weight_decay=0.01,
    extra_metrics={"cosine": cosine_similarity}
)


In [ ]:
trainer.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00001),
    # jit_compile=True,
)

trainer.fit(
    x_train,
    epochs=5,
    batch_size=16,
    verbose=1,
)

## 6. Evaluate Fine-tuned Model & Export

Check the accuracy of the fine-tuned Keras model, then export it back to
`.tflite` and verify that the Interpreter also shows improvement.

In [ ]:
# Evaluate the fine-tuned tunner model (Keras)
finetuned_logits = tunner_model.predict(x_test, verbose=0)
finetuned_preds = np.argmax(finetuned_logits, axis=-1)
finetuned_acc = np.mean(finetuned_preds == y_test)

print(f"Float32 teacher accuracy:         {float_acc:.4f}")
print(f"INT8 before fine-tuning:          {int8_acc:.4f}")
print(f"INT8 after fine-tuning (Keras):   {finetuned_acc:.4f}")

In [ ]:
# Export the fine-tuned model back to .tflite
finetuned_model_path = WORK_DIR / "mobilenetv2_int8_finetuned.tflite"
litert_tunner.save_model(tunner_model, str(finetuned_model_path))
print(f"Saved fine-tuned INT8 model to: {finetuned_model_path.name}")

# Evaluate via the LiteRT Interpreter
finetuned_int8_logits = run_tflite_inference(finetuned_model_path, x_test)
finetuned_int8_preds = np.argmax(finetuned_int8_logits, axis=-1)
finetuned_int8_acc = np.mean(finetuned_int8_preds == y_test)

print(f"\n{'='*50}")
print(f"📊 Final Results")
print(f"{'='*50}")
print(f"Float32 teacher accuracy:               {float_acc:.4f}")
print(f"INT8 before fine-tuning (Interpreter):  {int8_acc:.4f}")
print(f"INT8 after fine-tuning  (Interpreter):  {finetuned_int8_acc:.4f}")
print(f"{'='*50}")
print(f"Accuracy improvement:  {finetuned_int8_acc - int8_acc:+.4f}")
print(f"Gap to float teacher:  {float_acc - finetuned_int8_acc:+.4f}")

In [ ]:
testing_utils.cosine_similarity(finetuned_int8_logits, int8_logits_sample)


In [ ]:
# # Cleanup
# import shutil
# shutil.rmtree(WORK_DIR, ignore_errors=True)
# print(f"Cleaned up {WORK_DIR}")